# Imports/Settings

In [1]:
print("hello")

hello


In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
# 1. Стандартная библиотека
import os
import sys
import warnings
from pathlib import Path
import joblib

# --- ДИНАМИЧЕСКИЙ РАСЧЕТ АБСОЛЮТНЫХ ПУТЕЙ ---
notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "notebooks":
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from hydra import initialize, compose

# 3. Локальные модули
from core.data import UniversalDataLoader
from core.models import get_model
from core.utils import load_hydra_config

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
load_hydra_config.cache_clear()
cfg = load_hydra_config()

print(f"Проект: {cfg.project_name} | Режим: Error Analysis")

# --- НАСТРОЙКИ ВИЗУАЛИЗАЦИИ ---
try:
    p_cfg = cfg.logging.plots
    plt.style.use(p_cfg.style)
    plt.rcParams.update({
        'figure.figsize': p_cfg.fig_size,
        'figure.dpi': p_cfg.dpi,
        'font.size': p_cfg.font_size,
        'axes.grid': p_cfg.grid,
        'axes.spines.top': p_cfg.spines_top,
        'axes.spines.right': p_cfg.spines_right
    })
    PLOT_ALPHA = p_cfg.alpha
except AttributeError:
    PLOT_ALPHA = 0.5
    print("Используются дефолтные стили Matplotlib.")

Проект: outsource_project_name | Режим: Error Analysis


# Data Loading

In [ ]:
loader = UniversalDataLoader(cfg)
df = loader.load_data()
_, val_df, _ = loader.get_splits(df)

target = cfg.data.tabular.target_col
X_val, y_val = val_df.drop(columns=[target]), val_df[target]

print(f"Размер выборки для анализа: {X_val.shape}")

# --- ЗАГРУЗКА АРТЕФАКТОВ ИЗ ПАПКИ MODELS ---
models_dir = Path(cfg.paths.root_dir) / cfg.paths.models_dir
version = cfg.model.version
model_name = cfg.model.name

print("Загрузка препроцессора и весов модели...")
prep = joblib.load(models_dir / f"preprocessing_v{version}.pkl")

model = get_model(cfg)
model.load(str(models_dir / f"{model_name}_v{version}.cbm"))

# Errors Analise

In [ ]:
X_val_clean = prep.transform(X_val)
preds = model.predict(X_val_clean)

task_type = cfg.data.tabular.task_type

# Создаем удобный датафрейм для анализа
error_df = X_val.copy() # Берем сырые X_val, чтобы анализировать в понятных бизнес-терминах
error_df['Actual'] = y_val.values
error_df['Predicted'] = preds

if task_type == 'regression':
    # Для регрессии считаем абсолютную ошибку и остатки (Residuals)
    error_df['Error'] = error_df['Predicted'] - error_df['Actual']
    error_df['AbsError'] = error_df['Error'].abs()

elif task_type in ['binary', 'multiclass']:
    # Для классификации смотрим, угадал/не угадал
    error_df['Is_Error'] = (error_df['Actual'] != error_df['Predicted']).astype(int)

    if hasattr(model, 'predict_proba') and task_type == 'binary':
        error_df['Probability'] = model.predict_proba(X_val_clean)[:, 1]

error_df.head()

# Search Trends

In [ ]:
# ПРИМЕР: Анализ ошибки в зависимости от конкретной колонки
feature_to_analyze = 'total_hits' # Замени на нужную (например, device_category)

if feature_to_analyze in error_df.columns:
    plt.figure()

    if task_type == 'regression':
        # Диаграмма рассеяния: Ошибка vs Признак
        sns.scatterplot(data=error_df, x=feature_to_analyze, y='Error', alpha=PLOT_ALPHA)
        plt.axhline(0, color='red', linestyle='--')
        plt.title(f"Распределение ошибок (Residuals) по признаку: {feature_to_analyze}")

    elif task_type == 'binary':
        # Доля ошибок в зависимости от признака
        sns.histplot(
            data=error_df,
            x=feature_to_analyze,
            hue='Is_Error',
            multiple='fill', # Показывает процентное соотношение
            bins=20
        )
        plt.title(f"Доля ошибочных классификаций по признаку: {feature_to_analyze}")
        plt.ylabel("Процент ошибок")

    plt.show()

# Worst Preds

In [ ]:
print("=== ТОП-10 САМЫХ ГРУБЫХ ОШИБОК МОДЕЛИ ===")

if task_type == 'regression':
    # Сортируем по максимальной абсолютной ошибке
    worst_cases = error_df.sort_values(by='AbsError', ascending=False).head(10)

elif task_type == 'binary':
    # Ищем случаи, где модель была максимально уверена, но ошиблась
    # (False Positives с вероятностью 0.99 или False Negatives с вероятностью 0.01)
    if 'Probability' in error_df.columns:
        error_df['Confidence_Mistake'] = np.where(
            error_df['Actual'] == 1,
            1 - error_df['Probability'], # Если факт 1, модель ошиблась, если вероятность близка к 0
            error_df['Probability']      # Если факт 0, модель ошиблась, если вероятность близка к 1
        )
        worst_cases = error_df[error_df['Is_Error'] == 1].sort_values(by='Confidence_Mistake', ascending=False).head(10)
    else:
        worst_cases = error_df[error_df['Is_Error'] == 1].head(10)

display(worst_cases)